In [31]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
import operator
from pydantic import BaseModel, Field

In [32]:
load_dotenv()

True

In [33]:
generator_llm = ChatOpenAI(model="gpt-4o-mini") 
evaluator_llm = ChatOpenAI(model="gpt-4o-mini") 
optimizer_llm = ChatOpenAI(model="gpt-4o-mini") 

In [34]:
class PostEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the Facebook post.")

In [35]:
structured_evaluator_llm = evaluator_llm.with_structured_output(PostEvaluation)

In [36]:
post = """
Most people are still using AI like a glorified search engine or a writing assistant. But the real shift happening right now isn’t just smarter chatbots—it’s Agentic AI.   Here’s the difference:• Generative AI writes an email, summarizes a document, or drafts code when you ask.• Agentic AI takes a high-level goal, creates its own multi-step plan, uses external tools, navigates software, solves roadblocks, and gets the job done autonomously.   Instead of prompting:"Write an email following up with this lead."You tell an agent:"Research this prospect, check our CRM for past interactions, draft a tailored proposal, and schedule a call on my calendar once they confirm."   And it executes the sequence from start to finish.   We’re moving from AI as a tool you type questions into, toward AI teammates that handle end-to-end workflows in the background.   Are you already experimenting with autonomous AI agents in your daily workflow, or are you still sticking to traditional chat prompts? Drop your thoughts below! 👇#AgenticAI #ArtificialIntelligence #FutureOfWork #Automation #TechTrends
"""

In [37]:
result = structured_evaluator_llm.invoke(post)

In [38]:
result

PostEvaluation(evaluation='approved', feedback='This post provides a clear and engaging explanation of the difference between Generative AI and Agentic AI, highlighting the capabilities and future potential of the latter. The call to action encourages interaction, which is great for engagement. The use of hashtags is relevant and well-planned, helping to reach a wider audience interested in technology trends.')

In [39]:
# state
class PostState(TypedDict):
    topic: str 
    post: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

    post_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]

In [40]:
def generate_post(state: PostState):

    # Prompt
    messages = [
        SystemMessage(content="You are a funny and cleaver Facebook influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious Facebook post on the topic: "{state['topic']}".capitalize

Rules:
- Do NOT use question-answer format.
- Max 50 characters.
- Use observation humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    # Send generator_llm
    response = generator_llm.invoke(messages).content

    # return response
    return {'post': response, 'post_history': [response]}

In [41]:
def evaluate_post(state: PostState):

    # prompt
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Facebook critic. You evaluate posts based on humor, originality, virality, and post format."),
    HumanMessage(content=f"""
Evaluate the following Facebook post:

Post: "{state['post']}"

Use the criteria below to evaluate the post:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people share, react, or comment on it?  
5. Format – Is it a well-formed Facebook post (not a setup-punchline joke, not a Q&A joke, and under 500 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 500 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]

    response = structured_evaluator_llm.invoke(messages)

    return {'evaluation':response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [42]:
def optimize_post(state: PostState):

    messages = [
        SystemMessage(content="You punch up Facebook posts for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the Facebook post based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Post:
{state['post']}

Re-write it as a short, viral-worthy Facebook post. Avoid Q&A style and stay under 500 characters.
""")
    ]

    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration'] + 1

    return {'post': response, 'iteration': iteration, 'post_history': [response]}

In [43]:
def route_evaluation(state: PostState):
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [44]:
graph = StateGraph(PostState)

# add nodes
graph.add_node('generate_post', generate_post)
graph.add_node('evaluate_post', evaluate_post)
graph.add_node('optimize_post', optimize_post)

# add edges
graph.add_edge(START, 'generate_post')
graph.add_edge('generate_post', 'evaluate_post')
graph.add_conditional_edges('evaluate_post', route_evaluation,
        {'approved': END, 'needs_improvement': 'optimize_post'})
graph.add_edge('optimize_post', 'evaluate_post')

workflow = graph.compile()


In [45]:
initial_state = {
    "topic": "agentic AI",
    "iteration": 1,
    "max_iteration": 5
}
result = workflow.invoke(initial_state)

In [46]:
result

{'topic': 'agentic AI',
 'post': 'Introducing AGENTIC AI: The only thing that can work harder than your coffee addiction! ☕️🤖 Say goodbye to coffee breaks and hello to endless productivity! It’s like having a hyper-caffeinated buddy who doesn’t need sleep, snacks, or small talk. Sure, it’s great at crunching numbers, but can it handle your friend’s viral cat video obsession? Only time will tell! 🐱✨ Join the revolution where real breaks are a thing of the past, and let AGENTIC AI brew up the future of work! #NoCoffeeNeeded #AIatWork #RobotRevolution',
 'evaluation': 'approved',
 'feedback': 'This post scores well on originality as it creatively combines the themes of AI and coffee addiction in a humorous way. The humor is light-hearted and engaging, particularly the comparison of AGENTIC AI to a hyper-caffeinated buddy, which effectively adds a relatable touch. The punchy format keeps it under the character limit while still delivering a strong message. Its clever play on tech tools and

In [47]:
for post in result['post_history']:
    print(post)

AGENTIC AI: Because who needs a coffee break? ☕️🤖
Introducing AGENTIC AI: The only thing that can work harder than your coffee addiction! ☕️🤖 Say goodbye to coffee breaks and hello to endless productivity! It’s like having a hyper-caffeinated buddy who doesn’t need sleep, snacks, or small talk. Sure, it’s great at crunching numbers, but can it handle your friend’s viral cat video obsession? Only time will tell! 🐱✨ Join the revolution where real breaks are a thing of the past, and let AGENTIC AI brew up the future of work! #NoCoffeeNeeded #AIatWork #RobotRevolution
